[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andyrdt/puzzles/blob/main/10_2026/starter_notebook.ipynb)

# Monthly Algorithmic Challenge — October 2026: Set Difference, Part 2

*Inspired by Callum McDougall's [ARENA Monthly Algorithmic Challenges](https://learn.arena.education/chapter1_transformer_interp/monthly_algorithmic/).*

## Task

Choose `K` distinct symbols to form `X` (`2 ≤ K ≤ 8`), and let `Y` be an independent random permutation of `X`. The model reads

```
[BOS] x1 … xK [SEP] y1 … yK [EOS]
```

and is trained, at each position from `SEP` through `yK`, to predict the distribution over the next token:

| Position | Target |
|---|---|
| `SEP` | uniform over `X` |
| `yi` (`i < K`) | uniform over the symbols of `X` not among `y1 … yi` |
| `yK` | `EOS` |

Every valid next symbol receives a higher logit than every invalid token, at every position, on all held-out sets tested. Reverse-engineer the algorithm it learned.

## The model

| | |
|---|---|
| **Symbols** | 16 (rendered as `a` – `p`) |
| **Vocab** | 19 tokens (16 symbols + `BOS` + `SEP` + `EOS`) |
| **Set size `K`** | 2 – 8, mixed during training |
| **Layers** | 2 |
| **Heads per layer** | 1 |
| **`d_model`** | 64 |
| **Parameters** | 35,392 |
| **Positional information** | rotary embeddings (RoPE, base 10000) on queries and keys |
| **Normalization** | RMSNorm before each attention layer and before the unembedding |
| **Causal mask** | yes |

Attention-only: no MLPs, no biases.

```
token_embedding
    → [RMSNorm → attention (1 head, RoPE, causal) → + residual]  × 2
    → RMSNorm → linear unembed → logits
```

## Submission

Submit a Colab notebook describing the algorithm.

## Setup

In [ ]:
%pip install -q torch einops nnsight==0.6.3 huggingface_hub matplotlib

In [ ]:
import json, importlib, random
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from nnsight import NNsight
from huggingface_hub import hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

REPO_ID = "andyrdt/10_2026_puzzle_1"
model_py_path = hf_hub_download(REPO_ID, "model.py")
spec = importlib.util.spec_from_file_location("model", model_py_path)
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
Transformer = model_module.Transformer

config_path  = hf_hub_download(REPO_ID, "config.json")
weights_path = hf_hub_download(REPO_ID, "model.pt")

config = json.loads(Path(config_path).read_text())
raw_model = Transformer.from_config(config["model"])
raw_model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
raw_model.eval().to(device)
model = NNsight(raw_model)

print(f"Model config: {config['model']}")
print(f"Parameters: {sum(p.numel() for p in raw_model.parameters()):,}")

## Vocabulary

| Token id | Role | Render |
|---|---|---|
| `0` – `15` | Symbols | `a` – `p` |
| `16` | `BOS` | `BOS` |
| `17` | `SEP` | `SEP` |
| `18` | `EOS` | `EOS` |

A sequence with `K` symbols has length `2K + 3`.

In [ ]:
NUM_SYMBOLS = config["vocab"]["num_symbols"]   # 16
BOS, SEP, EOS = config["vocab"]["BOS"], config["vocab"]["SEP"], config["vocab"]["EOS"]

def symbol_to_token(s):
    return ord(s) - ord("a") if isinstance(s, str) else int(s)

def token_label(tok):
    if 0 <= tok < NUM_SYMBOLS:
        return chr(ord("a") + tok)
    return {BOS: "BOS", SEP: "SEP", EOS: "EOS"}.get(tok, f"?{tok}")

def encode(X, Y):
    """X and Y are lists of symbols (letters or ids); Y must be a permutation of X."""
    assert sorted(map(symbol_to_token, X)) == sorted(map(symbol_to_token, Y)) and len(set(X)) == len(X)
    return [BOS] + [symbol_to_token(s) for s in X] + [SEP] + [symbol_to_token(s) for s in Y] + [EOS]

def make_prompt(X, seed=0):
    """Shuffle X and an independent copy Y, then encode."""
    rng = random.Random(seed)
    X = list(X); Y = list(X)
    rng.shuffle(X); rng.shuffle(Y)
    return encode(X, Y)

def target_distribution(tokens):
    """Training target at each scored position: dict position -> {token: probability}."""
    K = (len(tokens) - 3) // 2
    X = tokens[1 : 1 + K]
    targets = {}
    for pos in range(K + 1, 2 * K + 2):              # SEP, y1, ..., yK
        seen = set(tokens[K + 2 : pos + 1])
        remaining = [s for s in X if s not in seen]
        targets[pos] = {s: 1 / len(remaining) for s in remaining} if remaining else {EOS: 1.0}
    return targets

tokens = make_prompt(["b", "e", "j", "n", "c"], seed=0)
print("token ids :", tokens)
print("labels    :", [token_label(t) for t in tokens])
for pos, tgt in target_distribution(tokens).items():
    print(f"  after {token_label(tokens[pos]):>3} (pos {pos:>2}) -> {sorted(token_label(t) for t in tgt)}")

## Running the model with `nnsight`

[`nnsight`](https://nnsight.net/) lets you capture activations and run interventions inside a `with model.trace(x): ...` block:

```python
with model.trace(x):
    out = model.output.save()
logits, attn_patterns = out    # attn_patterns[l] has shape (batch, n_heads, query, key)
```

Intermediate activations work the same way. For example, `model.blocks[0].output[0]` is the residual stream after layer 0, and `model.blocks[1].ln_attn.output` is the normalized input to layer 1's attention. Access modules in the order they execute. Interventions are in-place mutations of activation tensors inside the block.

In [ ]:
x = torch.tensor([tokens], device=device)
with model.trace(x):
    resid_after_layer0 = model.blocks[0].output[0].save()   # access modules in execution order
    out = model.output.save()
logits, attn_patterns = out
print("logits:", tuple(logits.shape), "| residual after layer 0:", tuple(resid_after_layer0.shape))

## Example outputs

Each column is a position in the sequence; the column shows the model's predicted distribution over the next token after reading that token. Positions before `SEP` are not scored and are left blank.

In [ ]:
import numpy as np

def plot_predictions(tokens):
    x = torch.tensor([tokens], device=device)
    with model.trace(x):
        out = model.output.save()
    logits, _ = out
    K = (len(tokens) - 3) // 2
    rows = list(range(NUM_SYMBOLS)) + [EOS]                  # predicted token
    probs = torch.softmax(logits[0].float(), dim=-1)[:, rows].T.detach().cpu().numpy()  # (vocab, seq)
    probs[:, : K + 1] = np.nan                               # unscored positions
    probs[:, 2 * K + 2 :] = np.nan

    fig, ax = plt.subplots(figsize=(0.55 * len(tokens) + 2, 0.32 * len(rows) + 1.2))
    cmap = plt.get_cmap("Blues").copy(); cmap.set_bad("white")
    ax.imshow(probs, vmin=0, vmax=1, cmap=cmap, aspect="auto")
    for (r, c), p in np.ndenumerate(probs):
        if not np.isnan(p) and p >= 0.01:
            ax.text(c, r, f"{p:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if p > 0.5 else "black")
    ax.set_xticks(range(len(tokens)), [token_label(t) for t in tokens])
    ax.set_yticks(range(len(rows)), [token_label(t) for t in rows])
    ax.set_xlabel("input token (position)"); ax.set_ylabel("predicted next token")
    ax.set_title(" ".join(token_label(t) for t in tokens), fontsize=10)
    plt.tight_layout()
    plt.show()

plot_predictions(make_prompt(["b", "e", "j", "n", "c"], seed=0))
plot_predictions(make_prompt(["a", "d", "f", "h", "k", "m", "o", "p"], seed=1))

## Attention patterns

Full attention pattern of each layer for one example.

In [ ]:
tokens = make_prompt(["b", "e", "j", "n", "c"], seed=0)
labels = [token_label(t) for t in tokens]
x = torch.tensor([tokens], device=device)
with model.trace(x):
    out = model.output.save()
_, attn_patterns = out

fig, axes = plt.subplots(1, len(attn_patterns), figsize=(5 * len(attn_patterns), 4.5))
for layer, ax in enumerate(axes):
    pattern = attn_patterns[layer][0, 0].detach().cpu()   # head 0: (query, key)
    ax.imshow(pattern, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=90)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("key"); ax.set_ylabel("query")
    ax.set_title(f"Layer {layer}")
plt.tight_layout()
plt.show()

## Weights

Direct access to every weight matrix via `raw_model`.

In [ ]:
for name, p in raw_model.named_parameters():
    print(f"  {name:32s}  shape = {tuple(p.shape)}")

## Your turn

- `model`: `nnsight`-wrapped; use `with model.trace(x): ...` to capture or intervene.
- `raw_model`: plain `nn.Module`; weights accessible directly.
- `make_prompt`, `encode`, `target_distribution`: build inputs and expected outputs.